# Step 4: Transformer

## 4.1 Ne Değişiyor?

Bu bölümde tek başlıklı attention yerine dört başlıklı
attention kullanacağız. Ayrıca modeli birden fazla
katmanla çalışabilecek şekilde düzenleyeceğiz.

Başlangıçta bir katman kullanacağız. Her katmanın
kendi parametreleri ve KV cache'i olacak.

In [1]:
import random
import math

random.seed(42)

with open("../data/raw_names.txt", encoding="utf-8") as f:
    docs = [
        line.strip()
        .replace("İ", "i")
        .replace("I", "ı")
        .lower()
        .replace(" ", "")
        for line in f
        if line.strip()
    ]

random.shuffle(docs)

# Verideki benzersiz karakterler
uchars = sorted(set("".join(docs)))

# Karakter → token numarası
stoi = {ch: i for i, ch in enumerate(uchars)}

# Başlangıç ve bitiş için özel token
BOS = len(uchars)
vocab_size = len(uchars) + 1

print("İsim sayısı:", len(docs))
print("Sözlük boyutu:", vocab_size)
print("BOS tokenı:", BOS)
print("Karakterler:", "".join(uchars))

İsim sayısı: 1100
Sözlük boyutu: 30
BOS tokenı: 29
Karakterler: abcdefghijklmnoprstuvyzçöüğış


In [2]:
class Value:
    __slots__ = ("data", "grad", "_children", "_local_grads")

    def __init__(self, data, children=(), local_grads=()):
        self.data = float(data)
        self.grad = 0.0
        self._children = children
        self._local_grads = local_grads

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)

        return Value(
            self.data + other.data,
            (self, other),
            (1.0, 1.0),
        )

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)

        return Value(
            self.data * other.data,
            (self, other),
            (other.data, self.data),
        )

    def __pow__(self, power):
        return Value(
            self.data ** power,
            (self,),
            (power * self.data ** (power - 1),),
        )

    def log(self):
        return Value(
            math.log(self.data),
            (self,),
            (1.0 / self.data,),
        )

    def exp(self):
        result = math.exp(self.data)

        return Value(
            result,
            (self,),
            (result,),
        )

    def relu(self):
        return Value(
            max(0.0, self.data),
            (self,),
            (float(self.data > 0),),
        )

    def __neg__(self):
        return self * -1

    def __radd__(self, other):
        return self + other

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * other ** -1

    def __rtruediv__(self, other):
        return other * self ** -1

    def backward(self):
        topo = []
        visited = set()

        def build_topo(node):
            if node not in visited:
                visited.add(node)

                for child in node._children:
                    build_topo(child)

                topo.append(node)

        build_topo(self)

        self.grad = 1.0

        for node in reversed(topo):
            for child, local_grad in zip(
                node._children, node._local_grads
            ):
                child.grad += local_grad * node.grad

In [3]:
# y = x² + 2x
x = Value(3.0)
y = x * x + 2 * x

y.backward()

print("y değeri:", y.data)
print("x gradyanı:", x.grad)

assert math.isclose(y.data, 15.0)
assert math.isclose(x.grad, 8.0)

y değeri: 15.0
x gradyanı: 8.0


In [4]:
def linear(x, w):
    return [
        sum(wi * xi for wi, xi in zip(row, x))
        for row in w
    ]


def softmax(logits):
    max_val = max(val.data for val in logits)

    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)

    return [e / total for e in exps]


def rmsnorm(x):
    mean_square = sum(xi * xi for xi in x) / len(x)
    scale = (mean_square + 1e-5) ** -0.5

    return [xi * scale for xi in x]

In [5]:
n_embd = 16
block_size = 16

n_head = 4
n_layer = 1

# Embedding boyutu başlık sayısına tam bölünmeli.
assert n_embd % n_head == 0, (
    "n_embd, n_head sayısına tam bölünmeli."
)

head_dim = n_embd // n_head

print("Embedding boyutu:", n_embd)
print("Attention başlığı sayısı:", n_head)
print("Başlık başına boyut:", head_dim)
print("Katman sayısı:", n_layer)

Embedding boyutu: 16
Attention başlığı sayısı: 4
Başlık başına boyut: 4
Katman sayısı: 1


## 4.2 Çok Başlı Attention

16 boyutlu Q, K ve V vektörlerini dört başlığa ayıracağız.
Her başlık dört boyut kullanacak ve kendi attention
ağırlıklarını hesaplayacak.

Başlıkların dört boyutlu çıktıları yan yana birleştirilerek
yeniden 16 boyutlu bir vektör oluşturulacak.

Başlıkların neye dikkat edeceği eğitim sırasında öğrenilecek.

## 4.3 Çok Başlı Attention Kodu

Çok başlı attention hesabını bir fonksiyonda toplayacağız.

Fonksiyonun girdileri:

- q: Mevcut tokenın Q vektörü.
- cached_keys: Geçmiş ve mevcut tokenların K vektörleri.
- cached_values: Geçmiş ve mevcut tokenların V vektörleri.

Her başlık Q, K ve V'nin kendi dilimini kullanır.
Başlık içinde skorlar hesaplanır, softmax uygulanır ve
V vektörlerinin ağırlıklı toplamı alınır.

Başlık çıktıları extend ile yan yana eklenir.
Fonksiyon sonunda n_embd boyutunda bir vektör döner.

In [6]:
matrix = lambda nout, nin, std=0.08: [
    [Value(random.gauss(0, std)) for _ in range(nin)]
    for _ in range(nout)
]

state_dict = {
    'wte': matrix(vocab_size, n_embd),
    'wpe': matrix(block_size, n_embd),
    'lm_head': matrix(vocab_size, n_embd),
}

for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4 * n_embd, n_embd)
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4 * n_embd)

params = [
    p
    for mat in state_dict.values()
    for row in mat
    for p in row
]

In [7]:
def gpt(token_id, pos_id, keys, values):
    tok_emb = state_dict['wte'][token_id]
    pos_emb = state_dict['wpe'][pos_id]
    x = [t + p for t, p in zip(tok_emb, pos_emb)]

    for li in range(n_layer):
        # 1) Multi-head attention block
        x_residual = x
        x = rmsnorm(x)

        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])

        keys[li].append(k)
        values[li].append(v)

        x_attn = []

        for h in range(n_head):
            hs = h * head_dim

            q_h = q[hs:hs + head_dim]
            k_h = [ki[hs:hs + head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs + head_dim] for vi in values[li]]

            attn_logits = [
                sum(q_h[j] * k_h[t][j] for j in range(head_dim))
                / head_dim ** 0.5
                for t in range(len(k_h))
            ]

            attn_weights = softmax(attn_logits)

            head_out = [
                sum(
                    attn_weights[t] * v_h[t][j]
                    for t in range(len(v_h))
                )
                for j in range(head_dim)
            ]

            x_attn.extend(head_out)

        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]

        # 2) MLP block: önceki bölümdeki yapı
        x_residual = x
        x = rmsnorm(x)

        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])
        x = [xi.relu() for xi in x]
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])

        x = [a + b for a, b in zip(x, x_residual)]

    logits = linear(x, state_dict['lm_head'])
    return logits

## 4.4 Katman Döngüsü

Attention ve MLP blokları, n_layer kadar çalışan bir döngüye
yerleştirilir. Bir katmanın çıktısı sonraki katmanın girdisi olur.

Her katmanın kendine ait dört attention matrisi ve iki MLP
matrisi bulunur. Parametre adlarında katman numarası kullanılır.

wte, wpe ve lm_head matrisleri katman döngüsünün dışındadır.

Her blokta girdi saklanır, normalizasyon ve hesaplama yapılır,
ardından saklanan girdi sonuca eklenir.

Başlangıçta n_layer = 1 olduğu için tek katman çalışır.